<a href="https://www.kaggle.com/code/insanejsk/query-rewriter-train?scriptVersionId=327956203" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

T5-small Query Rewriter — Fine-tuning
====
Input : "rewrite query: {noisy}"  -> "{clean}"

Eval  : F1 eval on val set

In [16]:
!pip install sentencepiece -q

In [1]:
import os, json, math
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    T5ForConditionalGeneration,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from tqdm import tqdm

In [2]:
# Config
CONFIG = {
    "model_name"    : "t5-small",
    "train_path"    : "/kaggle/input/datasets/insanejsk/queryrewriter-synthetic-ms-marco/data/train.jsonl",
    "val_path"      : "/kaggle/input/datasets/insanejsk/queryrewriter-synthetic-ms-marco/data/val.jsonl",
    "output_dir"    : "/kaggle/working/query_rewriter_f1/",
    "max_input_len" : 64,    # "rewrite query: " + noisy query
    "max_target_len": 48,    # clean query
    "batch_size"    : 64,
    "epochs"        : 3,
    "lr"            : 3e-4,  # T5 standard: higher than BERT
    "warmup_ratio"  : 0.05,
    "eval_every"    : 2000,  # steps
    "log_every"     : 200,
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)

In [3]:
# Dataset
class RewriterDataset(Dataset):
    def __init__(self, path):
        self.data = []
        with open(path) as f:
            for line in f:
                obj = json.loads(line)
                self.data.append((obj["noisy"], obj["clean"]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [4]:
def make_collate(tokenizer, max_input, max_target):
    def collate(batch):
        noisy = ["rewrite query: " + b[0] for b in batch]
        clean = [b[1] for b in batch]
        enc = tokenizer(
            noisy, max_length=max_input,
            padding=True, truncation=True, return_tensors="pt"
        )
        dec = tokenizer(clean, max_length=max_target,
                padding=True, truncation=True, return_tensors="pt")
        labels = dec["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100  # ignore padding in loss
        return enc["input_ids"], enc["attention_mask"], labels
    return collate

In [5]:
# F1 Eval
@torch.no_grad()
def evaluate(model, loader, tokenizer, device, num_batches=50):
    """Token F1 eval on first num_batches of val set."""
    core = model.module if hasattr(model, "module") else model
    core.eval()
    total_f1 = 0.0
    total     = 0

    for i, (input_ids, attn_mask, labels) in enumerate(loader):
        if i >= num_batches:
            break
        input_ids = input_ids.to(device)
        attn_mask = attn_mask.to(device)

        preds = core.generate(
            input_ids=input_ids,
            attention_mask=attn_mask,
            max_new_tokens=48,
            num_beams=4,
        )

        pred_strs = tokenizer.batch_decode(preds, skip_special_tokens=True)

        label_ids = labels.clone()
        label_ids[label_ids == -100] = tokenizer.pad_token_id
        true_strs = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        for p, t in zip(pred_strs, true_strs):
            pred_tokens = p.strip().lower().split()
            true_tokens = t.strip().lower().split()

            if len(pred_tokens) == 0 and len(true_tokens) == 0:
                total_f1 += 1.0
            elif len(pred_tokens) == 0 or len(true_tokens) == 0:
                total_f1 += 0.0
            else:
                pred_counts = {}
                for tok in pred_tokens:
                    pred_counts[tok] = pred_counts.get(tok, 0) + 1
                true_counts = {}
                for tok in true_tokens:
                    true_counts[tok] = true_counts.get(tok, 0) + 1

                overlap = sum(min(pred_counts.get(tok, 0), true_counts[tok])
                              for tok in true_counts)

                precision = overlap / len(pred_tokens)
                recall    = overlap / len(true_tokens)

                if precision + recall == 0:
                    total_f1 += 0.0
                else:
                    total_f1 += 2 * precision * recall / (precision + recall)

            total += 1

    core.train()
    return total_f1 / total if total > 0 else 0.0

In [6]:
# Training block
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device} | GPUs: {torch.cuda.device_count()}")

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
    model     = T5ForConditionalGeneration.from_pretrained(CONFIG["model_name"])

    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)
    model.to(device)

    collate   = make_collate(tokenizer, CONFIG["max_input_len"], CONFIG["max_target_len"])
    train_ds  = RewriterDataset(CONFIG["train_path"])
    val_ds    = RewriterDataset(CONFIG["val_path"])
    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"],
                               shuffle=True,  collate_fn=collate, num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"],
                               shuffle=False, collate_fn=collate, num_workers=2)

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
    total_steps   = len(train_loader) * CONFIG["epochs"]
    warmup_steps  = int(total_steps * CONFIG["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler    = torch.amp.GradScaler("cuda")

    best_em, global_step = 0.0, 0

    for epoch in range(1, CONFIG["epochs"] + 1):
        model.train()
        loop = tqdm(train_loader, desc=f"Epoch {epoch}")
        running_loss = 0.0

        for input_ids, attn_mask, labels in loop:
            input_ids = input_ids.to(device)
            attn_mask = attn_mask.to(device)
            labels    = labels.to(device)

            with torch.amp.autocast("cuda"):
                out  = model(input_ids=input_ids,
                             attention_mask=attn_mask,
                             labels=labels)
                loss = out.loss
                if isinstance(loss, torch.Tensor) and loss.dim() > 0:
                    loss = loss.mean()   # DataParallel returns per-GPU losses

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1

            running_loss += loss.item()
            if global_step % CONFIG["log_every"] == 0:
                avg = running_loss / CONFIG["log_every"]
                loop.set_postfix(loss=f"{avg:.4f}", step=global_step)
                running_loss = 0.0

            if global_step % CONFIG["eval_every"] == 0:
                em = evaluate(model, val_loader, tokenizer, device)
                print(f"\nStep {global_step} | F1: {em:.4f}")
                core = model.module if hasattr(model, "module") else model
                if em > best_em:
                    best_em = em
                    core.save_pretrained(CONFIG["output_dir"])
                    tokenizer.save_pretrained(CONFIG["output_dir"])
                    print(f"New best F1: {best_em:.4f} — saved")
                model.train()

        # End-of-epoch eval
        em = evaluate(model, val_loader, tokenizer, device)
        print(f"\nEpoch {epoch} complete | F1: {em:.4f} | Best: {best_em:.4f}")
        core = model.module if hasattr(model, "module") else model
        if em > best_em:
            best_em = em
            core.save_pretrained(CONFIG["output_dir"])
            tokenizer.save_pretrained(CONFIG["output_dir"])
            print(f"New best F1: {best_em:.4f} — saved")

    print(f"\nTraining complete. Best F1: {best_em:.4f}")
    print(f"Model saved to: {CONFIG['output_dir']}")

In [ ]:
train()

Query Rewriter Pipeline Eval
============================================================
Compares Recall@10 across 3 conditions:
   1. Raw query         → bi-encoder
   2. EM rewriter       → bi-encoder
   3. F1 rewriter       → bi-encoder

 Uses full val corpus (~9,944 passages)


In [18]:
# Config
CONFIG = {
    # Bi-encoder
    "biencoder_path"   : "/kaggle/input/models/insanejsk/denseretriever/pytorch/default/1/best_biencoder",
    "proj_dim"         : 256,
    "max_length"       : 180,

    # Rewriters
    "rewriter_f1_path" : "/kaggle/input/models/insanejsk/queryrewriter-finetuned-t5-small/pytorch/default/1/query_rewriter_f1",
    "max_input_len"    : 64,
    "max_output_len"   : 48,
    "num_beams"        : 4,

    # Val data
    "triples_val_path" : "/kaggle/input/datasets/insanejsk/nse-dr/triples_val.jsonl",
    "noisy_val_path"   : "/kaggle/input/datasets/insanejsk/queryrewriter-synthetic-ms-marco/data/val.jsonl",
    # Eval settings
    "num_queries"      : 5000,
    "top_k"            : 10,
    "batch_size"       : 64,
}

In [19]:
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, T5ForConditionalGeneration
# BiEncoder
class BiEncoder(nn.Module):
    def __init__(self, model_name, proj_dim=256):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.proj = nn.Sequential(
            nn.Linear(hidden, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim),
        )

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        mask = attention_mask.unsqueeze(-1).float()
        emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        emb = self.proj(emb)
        emb = nn.functional.normalize(emb, dim=-1)
        return emb

def load_biencoder(path, device):
    from safetensors.torch import load_file
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = BiEncoder(path, proj_dim=CONFIG["proj_dim"])
    proj_state = load_file(os.path.join(path, "proj.safetensors"), device=str(device))
    model.proj.load_state_dict(proj_state)
    model.to(device).eval()
    return model, tokenizer

In [20]:
# Rewriter
def load_rewriter(path, device):
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = T5ForConditionalGeneration.from_pretrained(path)
    model.to(device).eval()
    return model, tokenizer

In [21]:
@torch.no_grad()
def rewrite_queries(queries, model, tokenizer, device):
    results = []
    for start in range(0, len(queries), CONFIG["batch_size"]):
        batch = queries[start : start + CONFIG["batch_size"]]
        inputs = ["rewrite query: " + q for q in batch]
        enc = tokenizer(
            inputs,
            max_length=CONFIG["max_input_len"],
            padding=True, truncation=True, return_tensors="pt"
        ).to(device)
        preds = model.generate(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            max_new_tokens=CONFIG["max_output_len"],
            num_beams=CONFIG["num_beams"],
        )
        results.extend(tokenizer.batch_decode(preds, skip_special_tokens=True))
    return results

In [22]:
# Encode corpus / queries
@torch.no_grad()
def encode_texts(texts, model, tokenizer, device):
    all_embs = []
    for start in tqdm(range(0, len(texts), CONFIG["batch_size"]), desc="Encoding", leave=False):
        batch = texts[start : start + CONFIG["batch_size"]]
        enc = tokenizer(
            batch,
            max_length=CONFIG["max_length"],
            padding=True, truncation=True, return_tensors="pt"
        ).to(device)
        embs = model.encode(enc["input_ids"], enc["attention_mask"])
        all_embs.append(embs.cpu())
    return torch.cat(all_embs, dim=0)  # (N, 256)

In [23]:
# Recall@10
def recall_at_k(query_embs, corpus_embs, positive_indices, k=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    query_embs = query_embs.to(device)
    corpus_embs = corpus_embs.to(device)
    sims = query_embs @ corpus_embs.T
    topk = torch.topk(sims, k=k, dim=1).indices.cpu().tolist()
    hits = sum(int(pos_idx in topk[i]) for i, pos_idx in enumerate(positive_indices))
    return hits / len(positive_indices)

In [24]:
# Load val data
def load_val_data(triples_path, noisy_path, num_queries):
    # Build corpus from triples
    corpus, seen = [], set()
    passage_to_idx = {}
    triples = []
    with open(triples_path) as f:
        for line in f:
            triples.append(json.loads(line))

    for obj in triples:
        for passage in [obj["positive"], obj["negative"]]:
            if passage not in seen:
                passage_to_idx[passage] = len(corpus)
                corpus.append(passage)
                seen.add(passage)

    # Build noisy lookup keyed on clean query
    noisy_lookup = {}
    with open(noisy_path) as f:
        for line in f:
            obj = json.loads(line)
            noisy_lookup[obj["clean"]] = obj["noisy"]

    # Join — only keep queries that exist in both files
    queries, noisy_queries, positives = [], [], []
    for obj in triples:
        if len(queries) >= num_queries:
            break
        clean = obj["query"]
        if clean not in noisy_lookup:
            continue                        # skip if no noisy counterpart
        queries.append(clean)
        noisy_queries.append(noisy_lookup[clean])
        positives.append(passage_to_idx[obj["positive"]])

    print(f"Matched {len(queries):,} queries across both files")
    return queries, noisy_queries, corpus, positives

In [30]:
# Eval
def eval():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Load val data
    print("\nLoading val data...")
    queries, noisy_queries, corpus, positive_indices = load_val_data(
        CONFIG["triples_val_path"],
        CONFIG["noisy_val_path"],
        CONFIG["num_queries"]
    )
    print(f"Queries : {len(queries):,}")
    print(f"Corpus  : {len(corpus):,} passages")

    # Load bi-encoder
    print("\nLoading bi-encoder...")
    bi_model, bi_tokenizer = load_biencoder(CONFIG["biencoder_path"], device)

    # Encode corpus once shared across both conditions
    print("Encoding corpus...")
    corpus_embs = encode_texts(corpus, bi_model, bi_tokenizer, device)
    print(f"Corpus embeddings: {corpus_embs.shape}")

    results = {}

    # Condition 1: Raw queries
    print("\n[1/3] Raw queries → bi-encoder")
    raw_embs       = encode_texts(queries, bi_model, bi_tokenizer, device)
    results["raw"] = recall_at_k(raw_embs, corpus_embs, positive_indices, CONFIG["top_k"])
    print(f"  Recall@10: {results['raw']:.4f}")

    # Condition 2: Noisy queries direct
    print("\n[2/3] Noisy queries → bi-encoder")
    noisy_embs          = encode_texts(noisy_queries, bi_model, bi_tokenizer, device)
    results["noisy_raw"] = recall_at_k(noisy_embs, corpus_embs, positive_indices, CONFIG["top_k"])
    print(f"  Recall@10: {results['noisy_raw']:.4f}")
    
    # Condition 3: Noisy → F1 rewriter
    print("\n[3/3] Noisy queries → F1 rewriter → bi-encoder")
    f1_model, f1_tokenizer = load_rewriter(CONFIG["rewriter_f1_path"], device)
    print("  Rewriting queries...")
    f1_rewritten         = rewrite_queries(noisy_queries, f1_model, f1_tokenizer, device)
    f1_embs              = encode_texts(f1_rewritten, bi_model, bi_tokenizer, device)
    results["noisy_f1"]  = recall_at_k(f1_embs, corpus_embs, positive_indices, CONFIG["top_k"])
    print(f"  Recall@10: {results['noisy_f1']:.4f}")
    del f1_model
    
    # Summary
    print(f"{'Clean query (ceiling)':<25} {results['raw']:>10.4f}")
    print(f"{'Noisy direct':<25} {results['noisy_raw']:>10.4f}")
    print(f"{'Noisy + F1 rewriter':<25} {results['noisy_f1']:>10.4f}")
    print("="*50)
    print(f"\nRewriter recovery : {results['noisy_f1'] - results['noisy_raw']:+.4f}")
    print(f"Gap to ceiling    : {results['raw'] - results['noisy_f1']:+.4f}")
    
    # Delta vs raw
    print(f"F1 rewriter delta : {results['noisy_f1'] - results['raw']:+.4f}")

    # Sample rewrites for sanity check
    print("\nSample rewrites (first 5 queries):")
    print(f"{'Query':<40} {'F1 rewrite':<40}")
    print("-"*120)
    for q, f1 in zip(queries[:5], f1_rewritten[:5]):
        print(f"{q[:39]:<40} {f1[:39]:<40}")

In [31]:
eval()

Device: cuda

Loading val data...
Matched 5,000 queries across both files
Queries : 5,000
Corpus  : 9,945 passages

Loading bi-encoder...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding corpus...


Corpus embeddings: torch.Size([9945, 256])

[1/3] Raw queries → bi-encoder


  Recall@10: 0.9716

[2/3] Noisy queries → bi-encoder


  Recall@10: 0.5702

[3/3] Noisy queries → F1 rewriter → bi-encoder


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

  Rewriting queries...


  Recall@10: 0.6330
Clean query (ceiling)         0.9716
Noisy direct                  0.5702
Noisy + F1 rewriter           0.6330

Rewriter recovery : +0.0628
Gap to ceiling    : +0.3386
F1 rewriter delta : -0.3386

Sample rewrites (first 5 queries):
Query                                    F1 rewrite                              
------------------------------------------------------------------------------------------------------------------------
. what is a corporation?                 what is a corporatoon                   
why did rachel carson write an obligati  why did rahel carson have an obligation 
symptoms of a dying mouse                symptoms of a dying mouse               
average number of lightning strikes per  average number of lightning strikes per 
can you burn your lawn with fertilizer   can you burn your skin with ferilizer   


### Some notable points about the model

- The dataset we generated is very noisy, which causes severe degradation to performance
- (0.9716->0.5702) is a 41% degradation in results
- The current model has a recovery (0.5702 -> 0.6330) which is a 11% recovery
- The model being a very simple 60m parameter (T5-small) is incapable of solving such a taxing dataset
- The model does prove its benefits on purely noisy queries
- On clean queries, it may cause some actual loss as shown in the samples
    - Query: "can you burn your lawn with fertilizer" turned into
    - Rewrite: "can you burn your skin with ferilizer"